<a href="https://colab.research.google.com/github/tonHS/Canadian-Crime-Trends/blob/main/notebooks/Crime_Trends_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Canadian Crime Trends Analysis

## Overview
This notebook analyzes overall crime rate trends and retail crime in Canada using Statistics Canada data.

### Data Sources
- **Table 35-10-0177-01**: Incident-based crime statistics (Police-reported crime)
- **Table 35-10-0062-01**: Police-reported crime for selected offences and Criminal Code traffic violations

### Analysis Outputs
1. **Table**: Top 10 violent Criminal Code violations in 2024 with growth metrics
2. **Table**: Top 10 Criminal Code property violations in 2024 with growth metrics
3. **Visualizations**: Shoplifting trends (under/over $5,000) for both Criminal Code violations and Organized Crime

**Author**: tonHS  
**Date**: 2025-11-21

## Setup: Install Dependencies

In [ ]:
# Install required packages
!pip install pandas matplotlib requests openpyxl -q

import pandas as pd
import numpy as np
import requests
import zipfile
from io import BytesIO
from pathlib import Path
import matplotlib.pyplot as plt
from datetime import datetime

# Create directories
data_dir = Path('data')
outputs_dir = Path('outputs')
figures_dir = Path('figures')

for directory in [data_dir, outputs_dir, figures_dir]:
    directory.mkdir(exist_ok=True)

print("✓ Setup complete")

## Step 1: Fetch General Crime Data (Table 35-10-0177-01)

In [ ]:
print("=" * 80)
print("FETCHING GENERAL CRIME DATA (TABLE 35-10-0177-01)")
print("=" * 80)

# Table ID for incident-based crime statistics
TABLE_ID_GENERAL = "35100177"

# Download URL
api_url = f"https://www150.statcan.gc.ca/t1/wds/rest/getFullTableDownloadCSV/{TABLE_ID_GENERAL}/en"

try:
    print(f"\n📥 Requesting download URL from Statistics Canada API...")
    response = requests.get(api_url, timeout=60)
    
    if not response.ok:
        raise ValueError(f"API request failed: {response.status_code}")
    
    zip_url = response.json()['object']
    print(f"✓ Got download URL")
    
    # Download the ZIP file
    print(f"📥 Downloading data...")
    zip_response = requests.get(zip_url, timeout=60)
    
    if not zip_response.ok:
        raise ValueError(f"Data download failed: {zip_response.status_code}")
    
    # Extract and load the CSV
    with zipfile.ZipFile(BytesIO(zip_response.content)) as z:
        csv_filename = f"{TABLE_ID_GENERAL}.csv"
        if csv_filename not in z.namelist():
            csv_filename = [f for f in z.namelist() if f.endswith('.csv')][0]
        
        print(f"✓ Extracting: {csv_filename}")
        df_general = pd.read_csv(z.open(csv_filename), low_memory=False)
    
    print(f"✓ Data loaded successfully: {len(df_general):,} rows, {len(df_general.columns)} columns")
    
    # Save raw data
    raw_path = data_dir / 'general_crime_raw.csv'
    df_general.to_csv(raw_path, index=False)
    print(f"✓ Saved to: {raw_path}")
    
    # Display data info
    print(f"\n📊 Dataset Overview:")
    print(f"   • Columns: {', '.join(df_general.columns.tolist())}")
    print(f"   • Shape: {df_general.shape}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    raise

print("\n" + "=" * 80)
print("✓ DATA FETCH COMPLETE")
print("=" * 80)

## Step 2: Fetch Organized Crime Data (Table 35-10-0062-01)

In [ ]:
print("=" * 80)
print("FETCHING ORGANIZED CRIME DATA (TABLE 35-10-0062-01)")
print("=" * 80)

# Table ID for organized crime
TABLE_ID_ORGANIZED = "35100062"

# Use direct CSV download (as shown in Crime_Stats_MVP notebook)
download_url = f"https://www150.statcan.gc.ca/n1/tbl/csv/{TABLE_ID_ORGANIZED}-eng.zip"

try:
    print(f"\n📥 Downloading data from Statistics Canada...")
    print(f"URL: {download_url}")
    
    # Download the data
    response = requests.get(download_url, timeout=60)
    response.raise_for_status()
    
    # Extract ZIP file
    with zipfile.ZipFile(BytesIO(response.content)) as zip_file:
        csv_files = [f for f in zip_file.namelist() if f.endswith('.csv')]
        
        if not csv_files:
            raise ValueError("No CSV file found in downloaded ZIP")
        
        csv_filename = csv_files[0]
        print(f"✓ Downloaded and extracted: {csv_filename}")
        
        with zip_file.open(csv_filename) as csv_file:
            df_organized = pd.read_csv(csv_file, low_memory=False)
    
    print(f"✓ Data loaded successfully: {len(df_organized):,} rows, {len(df_organized.columns)} columns")
    
    # Save raw data
    raw_path = data_dir / 'organized_crime_raw.csv'
    df_organized.to_csv(raw_path, index=False)
    print(f"✓ Saved to: {raw_path}")
    
    # Display data info
    print(f"\n📊 Dataset Overview:")
    print(f"   • Time period: {df_organized['REF_DATE'].min()} to {df_organized['REF_DATE'].max()}")
    print(f"   • Columns: {', '.join(df_organized.columns.tolist())}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    raise

print("\n" + "=" * 80)
print("✓ DATA FETCH COMPLETE")
print("=" * 80)

## Step 3: Process and Clean Data

In [ ]:
print("=" * 80)
print("PROCESSING AND CLEANING DATA")
print("=" * 80)

# Process general crime data
df_general_clean = df_general.copy()
df_general_clean['Year'] = df_general_clean['REF_DATE'].astype(int)
df_general_clean = df_general_clean[df_general_clean['VALUE'].notna()]

# Find violation column
violation_cols = [col for col in df_general_clean.columns if 'violation' in col.lower()]
if violation_cols:
    violation_col_general = violation_cols[0]
    print(f"\n✓ General crime - Violation column: '{violation_col_general}'")
    print(f"  • Unique violations: {df_general_clean[violation_col_general].nunique():,}")
else:
    print("⚠ Warning: Could not find violation column in general crime data")
    violation_col_general = None

# Process organized crime data
df_organized_clean = df_organized.copy()
df_organized_clean['Year'] = df_organized_clean['REF_DATE'].astype(int)
df_organized_clean = df_organized_clean[df_organized_clean['VALUE'].notna()]

# Find violation column for organized crime
violation_cols_org = [col for col in df_organized_clean.columns if 'violation' in col.lower()]
if violation_cols_org:
    violation_col_organized = violation_cols_org[0]
    print(f"\n✓ Organized crime - Violation column: '{violation_col_organized}'")
    print(f"  • Unique violations: {df_organized_clean[violation_col_organized].nunique():,}")
else:
    print("⚠ Warning: Could not find violation column in organized crime data")
    violation_col_organized = None

# Display sample violations
if violation_col_general:
    print(f"\n📋 Sample general crime violations:")
    for i, v in enumerate(df_general_clean[violation_col_general].unique()[:10], 1):
        print(f"   {i}. {v}")

if violation_col_organized:
    print(f"\n📋 Sample organized crime violations:")
    for i, v in enumerate(df_organized_clean[violation_col_organized].unique()[:10], 1):
        print(f"   {i}. {v}")

print("\n" + "=" * 80)
print("✓ DATA PROCESSING COMPLETE")
print("=" * 80)

## Step 4: Analyze Top 10 Violent Criminal Code Violations (2024)

In [ ]:
print("=" * 80)
print("ANALYZING TOP 10 VIOLENT CRIMINAL CODE VIOLATIONS (2024)")
print("=" * 80)

# Keywords for identifying violent crimes
violent_keywords = [
    'homicide', 'murder', 'assault', 'sexual', 'robbery',
    'kidnapping', 'abduction', 'extortion', 'criminal harassment',
    'uttering threats', 'discharge firearm', 'weapons offence',
    'threatening', 'forcible', 'attempted murder'
]

# Filter for violent crimes
if violation_col_general:
    mask = df_general_clean[violation_col_general].str.lower().str.contains(
        '|'.join(violent_keywords), na=False
    )
    df_violent = df_general_clean[mask].copy()
    
    # Exclude total rows
    df_violent = df_violent[
        ~df_violent[violation_col_general].str.contains('Total', case=False, na=False)
    ]
    
    print(f"\n✓ Found {len(df_violent):,} records for violent crimes")
    
    # Calculate for 2024
    df_2024 = df_violent[df_violent['Year'] == 2024]
    violations_2024 = df_2024.groupby(violation_col_general)['VALUE'].sum().sort_values(ascending=False)
    
    # Get top 10
    top_10_violent = violations_2024.head(10)
    
    # Calculate growth metrics
    results = []
    for violation in top_10_violent.index:
        count_2024 = violations_2024.get(violation, 0)
        
        # Get 2019 data (5-year comparison)
        df_2019 = df_violent[df_violent['Year'] == 2019]
        count_2019 = df_2019[df_2019[violation_col_general] == violation]['VALUE'].sum()
        
        # Get 2014 data (10-year comparison)
        df_2014 = df_violent[df_violent['Year'] == 2014]
        count_2014 = df_2014[df_2014[violation_col_general] == violation]['VALUE'].sum()
        
        # Calculate growth rates
        growth_5yr = ((count_2024 - count_2019) / count_2019 * 100) if count_2019 > 0 else np.nan
        growth_10yr = ((count_2024 - count_2014) / count_2014 * 100) if count_2014 > 0 else np.nan
        
        results.append({
            'Rank': len(results) + 1,
            'Violation': violation,
            '2024 Count': int(count_2024),
            '5-Year Growth (%)': growth_5yr,
            '10-Year Growth (%)': growth_10yr
        })
    
    df_violent_results = pd.DataFrame(results)
    
    # Save to CSV
    output_path = outputs_dir / 'top_10_violent_crimes_2024.csv'
    df_violent_results.to_csv(output_path, index=False)
    print(f"\n✓ Results saved to: {output_path}")
    
    # Display formatted table
    df_display = df_violent_results.copy()
    df_display['2024 Count'] = df_display['2024 Count'].apply(lambda x: f'{x:,}')
    df_display['5-Year Growth (%)'] = df_display['5-Year Growth (%)'].apply(
        lambda x: f'{x:+.1f}%' if not pd.isna(x) else 'N/A'
    )
    df_display['10-Year Growth (%)'] = df_display['10-Year Growth (%)'].apply(
        lambda x: f'{x:+.1f}%' if not pd.isna(x) else 'N/A'
    )
    
    print("\n" + "=" * 80)
    print("TOP 10 VIOLENT CRIMINAL CODE VIOLATIONS (2024)")
    print("=" * 80)
    print(df_display.to_string(index=False))
    print("=" * 80)
    
    # Display as styled table
    from IPython.display import display
    display(df_display.style.set_properties(**{
        'text-align': 'left',
        'font-size': '11px'
    }).set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#2E86AB'), ('color', 'white'), ('font-weight', 'bold')]
    }]))
    
else:
    print("⚠ Cannot analyze - violation column not found")

print("\n✓ ANALYSIS COMPLETE")

## Step 5: Analyze Top 10 Property Criminal Code Violations (2024)

In [ ]:
print("=" * 80)
print("ANALYZING TOP 10 PROPERTY CRIMINAL CODE VIOLATIONS (2024)")
print("=" * 80)

# Keywords for identifying property crimes
property_keywords = [
    'theft', 'break and enter', 'fraud', 'mischief', 'arson',
    'shoplifting', 'motor vehicle theft', 'possession of stolen',
    'identity theft', 'identity fraud', 'forgery'
]

# Filter for property crimes
if violation_col_general:
    mask = df_general_clean[violation_col_general].str.lower().str.contains(
        '|'.join(property_keywords), na=False
    )
    df_property = df_general_clean[mask].copy()
    
    # Exclude total rows
    df_property = df_property[
        ~df_property[violation_col_general].str.contains('Total', case=False, na=False)
    ]
    
    print(f"\n✓ Found {len(df_property):,} records for property crimes")
    
    # Calculate for 2024
    df_2024 = df_property[df_property['Year'] == 2024]
    violations_2024 = df_2024.groupby(violation_col_general)['VALUE'].sum().sort_values(ascending=False)
    
    # Get top 10
    top_10_property = violations_2024.head(10)
    
    # Calculate growth metrics
    results = []
    for violation in top_10_property.index:
        count_2024 = violations_2024.get(violation, 0)
        
        # Get 2019 data (5-year comparison)
        df_2019 = df_property[df_property['Year'] == 2019]
        count_2019 = df_2019[df_2019[violation_col_general] == violation]['VALUE'].sum()
        
        # Get 2014 data (10-year comparison)
        df_2014 = df_property[df_property['Year'] == 2014]
        count_2014 = df_2014[df_2014[violation_col_general] == violation]['VALUE'].sum()
        
        # Calculate growth rates
        growth_5yr = ((count_2024 - count_2019) / count_2019 * 100) if count_2019 > 0 else np.nan
        growth_10yr = ((count_2024 - count_2014) / count_2014 * 100) if count_2014 > 0 else np.nan
        
        results.append({
            'Rank': len(results) + 1,
            'Violation': violation,
            '2024 Count': int(count_2024),
            '5-Year Growth (%)': growth_5yr,
            '10-Year Growth (%)': growth_10yr
        })
    
    df_property_results = pd.DataFrame(results)
    
    # Save to CSV
    output_path = outputs_dir / 'top_10_property_crimes_2024.csv'
    df_property_results.to_csv(output_path, index=False)
    print(f"\n✓ Results saved to: {output_path}")
    
    # Display formatted table
    df_display = df_property_results.copy()
    df_display['2024 Count'] = df_display['2024 Count'].apply(lambda x: f'{x:,}')
    df_display['5-Year Growth (%)'] = df_display['5-Year Growth (%)'].apply(
        lambda x: f'{x:+.1f}%' if not pd.isna(x) else 'N/A'
    )
    df_display['10-Year Growth (%)'] = df_display['10-Year Growth (%)'].apply(
        lambda x: f'{x:+.1f}%' if not pd.isna(x) else 'N/A'
    )
    
    print("\n" + "=" * 80)
    print("TOP 10 PROPERTY CRIMINAL CODE VIOLATIONS (2024)")
    print("=" * 80)
    print(df_display.to_string(index=False))
    print("=" * 80)
    
    # Display as styled table
    from IPython.display import display
    display(df_display.style.set_properties(**{
        'text-align': 'left',
        'font-size': '11px'
    }).set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#A23B72'), ('color', 'white'), ('font-weight', 'bold')]
    }]))
    
else:
    print("⚠ Cannot analyze - violation column not found")

print("\n✓ ANALYSIS COMPLETE")

## Step 6: Create Shoplifting Trends Visualization

In [ ]:
print("=" * 80)
print("CREATING SHOPLIFTING TRENDS VISUALIZATION")
print("=" * 80)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Shoplifting Trends in Canada: Criminal Code Violations vs Organized Crime',
             fontsize=16, fontweight='bold', y=0.98)

# Colors
COLOR_UNDER = '#2E86AB'
COLOR_OVER = '#A23B72'

# Plot 1: Criminal Code Shoplifting
if violation_col_general:
    print("\n📊 Processing Criminal Code shoplifting data...")
    
    # Filter for shoplifting violations
    shoplifting_under = df_general_clean[
        df_general_clean[violation_col_general].str.contains(
            'Shoplifting.*5,000 or under', case=False, na=False, regex=True
        )
    ]
    
    shoplifting_over = df_general_clean[
        df_general_clean[violation_col_general].str.contains(
            'Shoplifting.*over.*5,000', case=False, na=False, regex=True
        )
    ]
    
    # Group by year
    under_5k = shoplifting_under.groupby('Year')['VALUE'].sum().sort_index()
    over_5k = shoplifting_over.groupby('Year')['VALUE'].sum().sort_index()
    
    # Plot
    if not under_5k.empty:
        ax1.plot(under_5k.index, under_5k.values, marker='o', linewidth=2.5,
                markersize=8, label='Under $5,000', color=COLOR_UNDER, zorder=3)
    if not over_5k.empty:
        ax1.plot(over_5k.index, over_5k.values, marker='s', linewidth=2.5,
                markersize=8, label='Over $5,000', color=COLOR_OVER, zorder=3)
    
    print(f"  ✓ Found {len(under_5k)} years of data for under $5,000")
    print(f"  ✓ Found {len(over_5k)} years of data for over $5,000")

ax1.set_title('Criminal Code Violations', fontsize=14, fontweight='bold', pad=15)
ax1.set_xlabel('Year', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Incidents', fontsize=12, fontweight='bold')
ax1.legend(fontsize=11, loc='best', framealpha=0.95)
ax1.grid(True, alpha=0.3, linestyle='--', zorder=1)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
ax1.set_facecolor('#F8F9FA')

# Plot 2: Organized Crime Shoplifting
if violation_col_organized:
    print("\n📊 Processing Organized Crime shoplifting data...")
    
    # Filter for shoplifting violations
    org_shoplifting_under = df_organized_clean[
        df_organized_clean[violation_col_organized].str.contains(
            'Shoplifting.*5,000 or under', case=False, na=False, regex=True
        )
    ]
    
    org_shoplifting_over = df_organized_clean[
        df_organized_clean[violation_col_organized].str.contains(
            'Shoplifting.*over.*5,000', case=False, na=False, regex=True
        )
    ]
    
    # Group by year
    org_under_5k = org_shoplifting_under.groupby('Year')['VALUE'].sum().sort_index()
    org_over_5k = org_shoplifting_over.groupby('Year')['VALUE'].sum().sort_index()
    
    # Plot
    if not org_under_5k.empty:
        ax2.plot(org_under_5k.index, org_under_5k.values, marker='o', linewidth=2.5,
                markersize=8, label='Under $5,000', color=COLOR_UNDER, zorder=3)
    if not org_over_5k.empty:
        ax2.plot(org_over_5k.index, org_over_5k.values, marker='s', linewidth=2.5,
                markersize=8, label='Over $5,000', color=COLOR_OVER, zorder=3)
    
    print(f"  ✓ Found {len(org_under_5k)} years of data for under $5,000")
    print(f"  ✓ Found {len(org_over_5k)} years of data for over $5,000")

ax2.set_title('Organized Crime', fontsize=14, fontweight='bold', pad=15)
ax2.set_xlabel('Year', fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of Incidents', fontsize=12, fontweight='bold')
ax2.legend(fontsize=11, loc='best', framealpha=0.95)
ax2.grid(True, alpha=0.3, linestyle='--', zorder=1)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
ax2.set_facecolor('#F8F9FA')

plt.tight_layout()

# Save figure
output_path = figures_dir / 'shoplifting_trends_comparison.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\n✓ Visualization saved to: {output_path}")

plt.show()

print("\n" + "=" * 80)
print("✓ VISUALIZATION COMPLETE")
print("=" * 80)

## Summary and Conclusion

In [ ]:
print("=" * 80)
print("ANALYSIS SUMMARY")
print("=" * 80)

print("\n📊 Outputs Generated:")
print("\n1. Top 10 Violent Criminal Code Violations (2024)")
print(f"   • File: {outputs_dir}/top_10_violent_crimes_2024.csv")

print("\n2. Top 10 Property Criminal Code Violations (2024)")
print(f"   • File: {outputs_dir}/top_10_property_crimes_2024.csv")

print("\n3. Shoplifting Trends Visualization")
print(f"   • File: {figures_dir}/shoplifting_trends_comparison.png")
print("   • Shows: Trends for shoplifting under/over $5,000")
print("   • Comparison: Criminal Code vs Organized Crime")

print("\n" + "=" * 80)
print("✓ ALL ANALYSIS TASKS COMPLETED SUCCESSFULLY")
print("=" * 80)

print(f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Author: tonHS")